# 数据操作

### 导入pytorch

In [31]:
import torch

### 张量表示一个由数值组成的数组，这个数字可能有多个维度

In [32]:
x = torch.arange(12)
x

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [33]:
# shape属性访问张量的形状和张量中元素的总数
x.shape

torch.Size([12])

In [34]:
x.numel()  # 标量，元素总数

12

In [35]:
# reshape 改变张量的形状
X = x.reshape(3,4)
X

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [36]:
# 全0、全1、其他常量的张量
torch.zeros((2,3,4))

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [37]:
torch.ones((2,3,4))

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

In [38]:
# 赋值张量
torch.tensor([[2,1,4,5], [1,2,3,4], [3,2,5,1], [4,3,2,1]])

tensor([[2, 1, 4, 5],
        [1, 2, 3, 4],
        [3, 2, 5, 1],
        [4, 3, 2, 1]])

常见的标准算术运算符（+、-、*、/和**）都会升级为按元素运算

In [39]:
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 3])
x+y, x-y, x*y, x/y, x**y

(tensor([ 3.,  4.,  6., 11.]),
 tensor([-1.,  0.,  2.,  5.]),
 tensor([ 2.,  4.,  8., 24.]),
 tensor([0.5000, 1.0000, 2.0000, 2.6667]),
 tensor([  1.,   4.,  16., 512.]))

按元素方式应用更多计算

In [40]:
torch.exp(x)

tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])

多个张量连结一起

In [41]:
X = torch.arange(12, dtype=torch.float32).reshape((3,4))
Y = torch.tensor([[2.0, 1, 4, 3], [1,2,3,4], [4,3,2,1]])
torch.cat((X,Y), dim=0), torch.cat((X, Y), dim=1)  # dim=0 行方向  dim=1 列方向

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

通过逻辑运算符构建张量

In [42]:
X == Y

tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])

对张量中的所有元素进行求和会产生一个只有一个元素的张量

In [43]:
X.sum()

tensor(66.)

即使形状不同，仍然会通过调用广播机制来执行按元素操作

In [44]:
a = torch.arange(3).reshape((3,1))
b = torch.arange(2).reshape((1,2))
a, b

(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]))

In [45]:
a + b

tensor([[0, 1],
        [1, 2],
        [2, 3]])

广播机制会将 a 的shape转为（3,2），将 b 的shape也转为（3,2）再进行相加

numpy 张量 转换

In [46]:
A = X.numpy()
B = torch.tensor(A)
type(A), type(B)

(numpy.ndarray, torch.Tensor)

将大小为1的张量转换为Python标量

In [47]:
a = torch.tensor([3.5])
a, a.item(), float(a), int(a)

(tensor([3.5000]), 3.5, 3.5, 3)

## 数据预处理

创建一个人工数据集，并存储在csv文件

In [48]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms, Alley, Price\n')  # 列名
    f.write('NA, Pave, 127500\n')  # 每行表示一个数据样本
    f.write('2, NA, 106000\n')
    f.write('4, NA, 178100\n')
    f.write('NA, NA, 140000\n')

In [49]:
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms  Alley   Price
0       NaN   Pave  127500
1       2.0     NA  106000
2       4.0     NA  178100
3       NaN     NA  140000


为了处理缺失数据，典型的方法包括插值和删除，最常用的是插值

In [50]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
# 清理列名和字符串列中的空格
inputs.columns = inputs.columns.str.strip()
for col in inputs.columns:
    if inputs[col].dtype == object or str(inputs[col].dtype) == 'str':
        inputs[col] = inputs[col].str.strip().replace('NA', float('nan'))
# 数值列用均值填充，非数值列用众数填充
numeric_mean = inputs.mean(numeric_only=True)
inputs = inputs.fillna(numeric_mean)
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


对于inputs中的类别值或离散值，我们将“NaN”视为一个类别

In [51]:
inputs = pd.get_dummies(inputs, dummy_na=True).astype(int)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0         3           1          0
1         2           0          1
2         4           0          1
3         3           0          1


In [52]:
import torch

X, y = torch.tensor(inputs.values), torch.tensor(outputs.values)

X, y 

(tensor([[3, 1, 0],
         [2, 0, 1],
         [4, 0, 1],
         [3, 0, 1]], dtype=torch.int32),
 tensor([127500, 106000, 178100, 140000]))